# Notebook 1 — MLflow Experiment Tracking with PyTorch

This notebook demonstrates **MLflow's experiment tracking** for a PyTorch CNN trained on MNIST.

**What you'll learn:**
- `mlflow.pytorch.autolog()` — capture params, metrics, and model with one line
- Explicit `mlflow.log_params()` / `mlflow.log_metrics()` for custom data
- Logging figures and artifacts
- Comparing multiple runs in the MLflow UI
- Searching runs programmatically with the `MlflowClient`

**Prerequisites:** Docker stack running + `uv sync` completed

In [ ]:
import sys
sys.path.insert(0, '../src')

import os
import mlflow
from dotenv import load_dotenv

load_dotenv()

tracking_uri = os.environ.get('MLFLOW_TRACKING_URI', 'http://localhost:5000')
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment('mnist-cnn')

print('MLflow version:', mlflow.__version__)
print('Tracking URI:  ', mlflow.get_tracking_uri())

## 1. Single Training Run

In [ ]:
from training.config import TrainingConfig
from training.train import train

cfg = TrainingConfig(
    epochs=3,
    learning_rate=1e-3,
    batch_size=64,
    experiment_name='mnist-cnn',
)

run_id = train(cfg)
print(f'\nCompleted run_id: {run_id}')
print('View at: http://localhost:5000')

## 2. Compare Multiple Configurations

In [ ]:
configurations = [
    {'learning_rate': 1e-2, 'optimizer': 'adam', 'scheduler': 'cosine'},
    {'learning_rate': 1e-3, 'optimizer': 'adam', 'scheduler': 'cosine'},
    {'learning_rate': 1e-3, 'optimizer': 'sgd',  'scheduler': 'step'},
]

run_ids = []
for config_overrides in configurations:
    cfg = TrainingConfig(epochs=2, experiment_name='mnist-cnn', **config_overrides)
    run_id = train(cfg)
    run_ids.append(run_id)
    print(f'  Completed: lr={config_overrides["learning_rate"]}, opt={config_overrides["optimizer"]}')

print(f'\nCompleted {len(run_ids)} runs. Compare at: http://localhost:5000')

## 3. Query Runs Programmatically

In [ ]:
import pandas as pd

client = mlflow.MlflowClient()
experiment = client.get_experiment_by_name('mnist-cnn')

if experiment:
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=['metrics.best_val_acc DESC'],
        max_results=10,
    )
    
    rows = []
    for run in runs:
        rows.append({
            'run_id': run.info.run_id[:8] + '...',
            'run_name': run.info.run_name,
            'lr': run.data.params.get('learning_rate', 'N/A'),
            'optimizer': run.data.params.get('optimizer', 'N/A'),
            'epochs': run.data.params.get('epochs', 'N/A'),
            'best_val_acc': round(run.data.metrics.get('best_val_acc', 0), 4),
            'val_loss': round(run.data.metrics.get('val_loss', 0), 4),
        })
    
    df = pd.DataFrame(rows)
    print('Top runs by validation accuracy:')
    display(df)
else:
    print('No experiment found yet — run a training job first.')

## 4. Filter Runs by Metric Threshold

In [ ]:
if experiment:
    good_runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string='metrics.best_val_acc > 0.97',
        order_by=['metrics.best_val_acc DESC'],
    )
    print(f'Runs with val_acc > 0.97: {len(good_runs)}')
    for r in good_runs:
        acc = r.data.metrics.get('best_val_acc', 0)
        print(f'  run_id={r.info.run_id[:12]}  val_acc={acc:.4f}  lr={r.data.params.get("learning_rate")}')

## Summary

What MLflow captured automatically with `mlflow.pytorch.autolog()`:
- All hyperparameters from the optimizer and scheduler
- Training and validation metrics per epoch
- Model architecture summary
- System info (CPU, GPU, Python version)

What we added manually:
- Custom params (dataset, model_name, device)
- Training curve figure
- Model with signature (enables type-safe serving)

Next: see **Notebook 2** for Model Registry and deployment.